# GWB iniettato con log10_A = -14.0 e gamma = 13/3

Dati simulati in `Simulations.ipynb` (seed `jax.random.key(1)`, cartella `sim_data_noiseGWB_A140_g433/`): red noise + DM + scattering + GWB con **log10_A = -14.0** e **gamma = 13/3 ≈ 4.333**, cioè lo spettro teorico delle binarie di buchi neri supermassicci.

Analisi con il solito modello a 6 parametri, **senza** termine GWB.

Serve a separare due effetti che nel notebook 06 erano mescolati:
- contro `chain_sim_gwb4` (A = -14.0, γ = 2.7) isola l'effetto della **pendenza**, a parità di ampiezza;
- contro `chain_sim_gwb2` (A = -14.5, γ = 13/3) isola l'effetto dell'**ampiezza**, a parità di pendenza.

## Setup comune

Copia delle celle 0, 1, 3, 4, 6, 7, 9 di `Likelihood.ipynb`.

In [ ]:
import numpy as np
import os, glob, json
import pickle

import sys

# this is the path to the discovery folder
sys.path.insert(0, "./")
import discovery_new as ds

import matplotlib.pyplot as plt
import scipy
import jax.numpy as jnp

from scipy.stats import uniform
import re
from PTMCMCSampler.PTMCMCSampler import PTSampler as ptmcmc
from scipy.stats import uniform
import jax

import time
import emcee

In [ ]:
# path to the folder with the data
path = './dati/DR2low_feathers/'
s_dsfiles = np.sort(os.listdir(path))
psr = [ds.pulsar.Pulsar.read_feather(path + f'{psrfile}') for psrfile in s_dsfiles][9] # we are only analysing pulsar #9

In [ ]:
# data
model = [psr.residuals]
# timing model
model += [ds.makegp_timing(psr, svd=True)]
# white noise parameters
model += [ds.makenoise_measurement(psr, psr.noisedict, tnequad = True, ecorr=True)]
# red noise paramters
model.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_red_components'], T=ds.getspan(psr), name='red_noise' ))
# DM variation parameters
model.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_dm_components'], T=ds.getspan(psr), name='dm_gp', fourierbasis=ds.make_dmfourierbasis(alpha = 2.0, tndm = False)))
# scattering noise parameters
model.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_chrom_components'], T=ds.getspan(psr), name='sv_gp', fourierbasis=ds.make_dmfourierbasis(alpha = 4.0, tndm = False)))

In [ ]:
logL = ds.PulsarLikelihood(model).logL

In [ ]:
# annoying function needed because the function logL want a dictionary as input, but our sampler wants a likelihood with an array as input
# so we have to switch from one to another :(
def params_dict_from_array(x):
    # ensure x is a plain numpy array before slicing
    x = np.asarray(x)
    return dict(zip(logL.params, x))

In [ ]:
##### Cell for the prior distributions ###################################################################################

# dictionary with minimum and maximum value of the prior distribution
my_priordict = {}
my_priordict["(.*_)?sv_gp_gamma.*"] = [0, 7]
my_priordict["(.*_)?sv_gp_alpha.*"] = [0, 10]
my_priordict["(.*_)?sv_gp_log10_A.*"] = [-20, -11]
my_priordict["(.*_)?sw_log10_A.*"] = [-10, 1]
my_priordict["(.*_)?sw_gamma.*"] = [-5, 6]
my_priordict["(.*_)?dm_gp_gamma.*"] = [0, 7]
my_priordict["(.*_)?dm_gp_log10_A.*"] = [-20, -11]
my_priordict["(.*_)?red_noise_gamma.*"] = [0, 7]
my_priordict["(.*_)?red_noise_log10_A.*"] = [-20, -11]
my_priordict["(.*_)?sw_n_earth.*"] = [0, 30]


def build_priors(param_names, priordict):
    """
    Build a {param_name: scipy dist} dict with uniform prior distributions in the range
    """
    priors = {}
    unmatched = []
    for name in param_names:
        matched = False
        for pattern, (lo, hi) in priordict.items():
            if re.fullmatch(pattern, name):
                priors[name] = uniform(loc=lo, scale=hi - lo) ## this is where the uniform distribution is defined
                matched = True
                break
        if not matched:
            unmatched.append(name)

    if unmatched:
        print(f"Warning: no prior found for {len(unmatched)} parameters, "
              f"falling back to standard normal:\n  {unmatched}")
    return priors

priors = build_priors(logL.params, my_priordict)
priors

def log_prior(x):
    """Compute log prior"""
    params = params_dict_from_array(x)
    
    logp = 0.0
    for name, value in params.items():
        if name in priors:
            logp += priors[name].logpdf(value)
        else:
            # Default: standard normal prior
            if isinstance(value, np.ndarray):
                logp += -0.5 * np.sum(value**2)
            else:
                logp += -0.5 * value**2
    
    return logp

def prior_draw():
    """Draw from prior and return as flat array"""
    x = []
    for name in logL.params:
        x.append(priors[name].rvs())
    return np.array(x)

x0 = prior_draw()
x0

In [ ]:
@jax.jit # compile the likelihood in jax to make it faster
def _jit_logL(params_array):
    params = dict(zip(logL.params, params_array))
    return logL(params)

_jit_logL(x0)

def log_likelihood(x): 
    """
    Likelihood function that is given to the samples, it uses the jitted (=faster) version of the likelihood
    """
    x = np.asarray(x, dtype=np.float64)
    result = _jit_logL(jnp.array(x))
    return float(np.array(result))  # always a plain Python float

## Catene di confronto (lette da disco, nessun sampling)

- `chain`: dati reali DR2low
- `chain_sim`: simulazione senza GWB, stesso seed `key(1)`
- `chain_sim_gwb4`: GWB con A = -14.0, γ = 2.7 (notebook 06)
- `chain_sim_gwb2`: GWB con A = -14.5, γ = 13/3 (notebook 04)

In [ ]:
def load_chain(path):
    c = np.loadtxt(path)[:, :-4]
    return c[len(c)//4:]           # rimuovo il 25% di burn-in

chain          = load_chain('./test/chain_1.txt')
chain_sim      = load_chain('./test_sim/chain_1.txt')
chain_sim_gwb4 = load_chain('./test_sim_gwb_A140_g27/chain_1.txt')
chain_sim_gwb2 = load_chain('./test_sim_gwb_A145/chain_1.txt')
print(chain.shape, chain_sim.shape, chain_sim_gwb4.shape, chain_sim_gwb2.shape)

## Dati simulati con GWB (log10_A = -14.0, gamma = 13/3)

In [ ]:
sim_gwb5_psr = ds.pulsar.Pulsar.read_feather('sim_data_noiseGWB_A140_g433/J1640+2224')
print('nome     :', sim_gwb5_psr.name)
print('N_TOA    :', len(sim_gwb5_psr.toas))
print('std residui:', np.std(sim_gwb5_psr.residuals))

In [ ]:
# stesso modello a 6 par dei dati reali (nessun termine GWB)
model_sim_gwb5 = [sim_gwb5_psr.residuals]
model_sim_gwb5 += [ds.makegp_timing(sim_gwb5_psr, svd=True)]
model_sim_gwb5 += [ds.makenoise_measurement(sim_gwb5_psr, sim_gwb5_psr.noisedict, tnequad=True, ecorr=True)]
model_sim_gwb5.append(ds.makegp_fourier(sim_gwb5_psr, ds.powerlaw, sim_gwb5_psr.noisedict[sim_gwb5_psr.name + '_red_components'],
                                        T=ds.getspan(sim_gwb5_psr), name='red_noise'))
model_sim_gwb5.append(ds.makegp_fourier(sim_gwb5_psr, ds.powerlaw, sim_gwb5_psr.noisedict[sim_gwb5_psr.name + '_dm_components'],
                                        T=ds.getspan(sim_gwb5_psr), name='dm_gp',
                                        fourierbasis=ds.make_dmfourierbasis(alpha=2.0, tndm=False)))
model_sim_gwb5.append(ds.makegp_fourier(sim_gwb5_psr, ds.powerlaw, sim_gwb5_psr.noisedict[sim_gwb5_psr.name + '_chrom_components'],
                                        T=ds.getspan(sim_gwb5_psr), name='sv_gp',
                                        fourierbasis=ds.make_dmfourierbasis(alpha=4.0, tndm=False)))

logL_sim_gwb5 = ds.PulsarLikelihood(model_sim_gwb5).logL
logL_sim_gwb5.params

In [ ]:
assert list(logL_sim_gwb5.params) == list(logL.params), (logL_sim_gwb5.params, logL.params)
print('OK: parametri identici a logL.params')

In [ ]:
def params_dict_from_array_sim_gwb5(x):
    x = np.asarray(x)
    return dict(zip(logL_sim_gwb5.params, x))

priors_sim_gwb5 = build_priors(logL_sim_gwb5.params, my_priordict)

def log_prior_sim_gwb5(x):
    params = params_dict_from_array_sim_gwb5(x)
    logp = 0.0
    for name, value in params.items():
        if name in priors_sim_gwb5:
            logp += priors_sim_gwb5[name].logpdf(value)
        else:
            logp += -0.5 * np.sum(np.asarray(value)**2)
    return logp

def prior_draw_sim_gwb5():
    return np.array([priors_sim_gwb5[name].rvs() for name in logL_sim_gwb5.params])

x0_sim_gwb5 = prior_draw_sim_gwb5()

In [ ]:
@jax.jit
def _jit_logL_sim_gwb5(params_array):
    return logL_sim_gwb5(dict(zip(logL_sim_gwb5.params, params_array)))

def log_likelihood_sim_gwb5(x):
    x = np.asarray(x, dtype=np.float64)
    return float(np.array(_jit_logL_sim_gwb5(jnp.array(x))))

print('logL iniziale =', log_likelihood_sim_gwb5(x0_sim_gwb5))

In [ ]:
sampler_sim_gwb5 = ptmcmc(
    ndim=len(logL_sim_gwb5.params),
    logl=log_likelihood_sim_gwb5,
    logp=log_prior_sim_gwb5,
    cov=np.eye(len(logL_sim_gwb5.params)) * 0.001,
    outDir='./test_sim_gwb_A140_g433/',
    resume=True
)

### Sampling

Eseguire **una sola volta**. Circa 2 ore. Con τ ≈ 6 bastano 2×10⁵ iterazioni, ma per confrontabilità con le altre iniezioni restano 10⁶.

In [ ]:
print("Starting sampling (sim CON GWB log10_A=-14.0, gamma=13/3, analizzata SENZA GWB)...")
sampler_sim_gwb5.sample(x0_sim_gwb5, Niter=1000000, thin=10)

## Catena e grafici

In [ ]:
chain_sim_gwb5 = load_chain('./test_sim_gwb_A140_g433/chain_1.txt')
chain_sim_gwb5.shape

In [ ]:
import emcee
tau5 = emcee.autocorr.integrated_time(chain_sim_gwb5, has_walkers=False, quiet=True)
print('tau =', np.round(tau5, 1))
print('Effective length =', int(len(chain_sim_gwb5) / tau5.max()))

In [ ]:
# tabella: quanto si sposta ogni parametro rispetto alla simulazione senza GWB
truths_dict = {
    'J1640+2224_dm_gp_gamma': 6.471070651479037,
    'J1640+2224_dm_gp_log10_A': -15.7227863246785,
    'J1640+2224_red_noise_gamma': 0.13235259553929019,
    'J1640+2224_red_noise_log10_A': -13.212902548624186,
    'J1640+2224_sv_gp_gamma': 1.3259815720503758,
    'J1640+2224_sv_gp_log10_A': -15.888300265911965,
}
truth_values = [truths_dict[p] for p in logL.params]
clean_labels = [name.replace('J1640+2224_', '') for name in logL.params]

print(f"{'parametro':20s} {'vero':>8s} | {'senza GWB':>20s} | {'GWB -14, 13/3':>20s} | scost/sigma")
for i, name in enumerate(logL.params):
    a16, a50, a84 = np.percentile(chain_sim[:, i], [16, 50, 84])
    b16, b50, b84 = np.percentile(chain_sim_gwb5[:, i], [16, 50, 84])
    s = np.sqrt(((a84 - a16)/2)**2 + ((b84 - b16)/2)**2)
    print(f"{name.replace('J1640+2224_', ''):20s} {truths_dict[name]:8.2f} | {a50:7.2f} [{a16:6.2f},{a84:6.2f}] | {b50:7.2f} [{b16:6.2f},{b84:6.2f}] | {(b50-a50)/s:6.2f}")

### Recupero dei parametri

In [ ]:
import corner

truths_dict = {
    'J1640+2224_dm_gp_gamma': 6.471070651479037,
    'J1640+2224_dm_gp_log10_A': -15.7227863246785,
    'J1640+2224_red_noise_gamma': 0.13235259553929019,
    'J1640+2224_red_noise_log10_A': -13.212902548624186,
    'J1640+2224_sv_gp_gamma': 1.3259815720503758,
    'J1640+2224_sv_gp_log10_A': -15.888300265911965,
}
truth_values = [truths_dict[p] for p in logL.params]
clean_labels = [name.replace('J1640+2224_', '') for name in logL.params]

fig = corner.corner(
    chain_sim_gwb5,
    labels=clean_labels,
    truths=truth_values,
    truth_color='#222222',
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 11},
    hist_kwargs={'density': True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    bins=40,
    color='#A06CD5',
)
fig.set_size_inches(16, 16)
fig.suptitle('J1640+2224 — sim CON GWB (log10_A=-14.0, γ=13/3), analizzata SENZA GWB\nlinee nere = valori iniettati dei soli processi di rumore', fontsize=14, y=1.00)
fig.savefig('corner_sim_gwb_A140_g433.png', dpi=150, bbox_inches='tight')

### Effetto della pendenza: γ = 13/3 vs γ = 2.7, a parità di ampiezza (A = −14.0)

In [ ]:
import corner

truths_dict = {
    'J1640+2224_dm_gp_gamma': 6.471070651479037,
    'J1640+2224_dm_gp_log10_A': -15.7227863246785,
    'J1640+2224_red_noise_gamma': 0.13235259553929019,
    'J1640+2224_red_noise_log10_A': -13.212902548624186,
    'J1640+2224_sv_gp_gamma': 1.3259815720503758,
    'J1640+2224_sv_gp_log10_A': -15.888300265911965,
}
truth_values = [truths_dict[p] for p in logL.params]
clean_labels = [name.replace('J1640+2224_', '') for name in logL.params]

# GWB A=-14.0, gamma=2.7 (notebook 06)
fig = corner.corner(
    chain_sim_gwb4,
    labels=clean_labels,
    truths=truth_values,
    truth_color='#222222',
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 11},
    hist_kwargs={'density': True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    bins=40,
    color='#6A4C93',
)

# GWB A=-14.0, gamma=13/3 (questo notebook)
corner.corner(
    chain_sim_gwb5,
    labels=clean_labels,
    truths=truth_values,
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 11},
    hist_kwargs={'density': True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    bins=40,
    color='#A06CD5',
    fig=fig
)

fig.set_size_inches(16, 16)
fig.suptitle('J1640+2224 — GWB con A=-14.0: γ=2.7 (viola) vs γ=13/3 (viola chiaro)', fontsize=13, y=1.00)
fig.savefig('corner_gwb_A140_g27_vs_g433.png', dpi=150, bbox_inches='tight')

### Effetto dell'ampiezza: A = −14.0 vs A = −14.5, a parità di pendenza (γ = 13/3)

In [ ]:
import corner

truths_dict = {
    'J1640+2224_dm_gp_gamma': 6.471070651479037,
    'J1640+2224_dm_gp_log10_A': -15.7227863246785,
    'J1640+2224_red_noise_gamma': 0.13235259553929019,
    'J1640+2224_red_noise_log10_A': -13.212902548624186,
    'J1640+2224_sv_gp_gamma': 1.3259815720503758,
    'J1640+2224_sv_gp_log10_A': -15.888300265911965,
}
truth_values = [truths_dict[p] for p in logL.params]
clean_labels = [name.replace('J1640+2224_', '') for name in logL.params]

# GWB A=-14.5, gamma=13/3 (notebook 04)
fig = corner.corner(
    chain_sim_gwb2,
    labels=clean_labels,
    truths=truth_values,
    truth_color='#222222',
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 11},
    hist_kwargs={'density': True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    bins=40,
    color='#6A4C93',
)

# GWB A=-14.0, gamma=13/3 (questo notebook)
corner.corner(
    chain_sim_gwb5,
    labels=clean_labels,
    truths=truth_values,
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 11},
    hist_kwargs={'density': True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    bins=40,
    color='#A06CD5',
    fig=fig
)

fig.set_size_inches(16, 16)
fig.suptitle('J1640+2224 — GWB con γ=13/3: A=-14.5 (viola) vs A=-14.0 (viola chiaro)', fontsize=13, y=1.00)
fig.savefig('corner_gwb_g433_A145_vs_A140.png', dpi=150, bbox_inches='tight')

### Confronto con la simulazione senza GWB e con i dati reali

In [ ]:
import corner

truths_dict = {
    'J1640+2224_dm_gp_gamma': 6.471070651479037,
    'J1640+2224_dm_gp_log10_A': -15.7227863246785,
    'J1640+2224_red_noise_gamma': 0.13235259553929019,
    'J1640+2224_red_noise_log10_A': -13.212902548624186,
    'J1640+2224_sv_gp_gamma': 1.3259815720503758,
    'J1640+2224_sv_gp_log10_A': -15.888300265911965,
}
truth_values = [truths_dict[p] for p in logL.params]
clean_labels = [name.replace('J1640+2224_', '') for name in logL.params]

# simulazione senza GWB, stesso seed
fig = corner.corner(
    chain_sim,
    labels=clean_labels,
    truths=truth_values,
    truth_color='#222222',
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 11},
    hist_kwargs={'density': True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    bins=40,
    color='#2A9D8F',
)

# GWB A=-14.0, gamma=13/3
corner.corner(
    chain_sim_gwb5,
    labels=clean_labels,
    truths=truth_values,
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 11},
    hist_kwargs={'density': True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    bins=40,
    color='#A06CD5',
    fig=fig
)

fig.set_size_inches(16, 16)
fig.suptitle('J1640+2224 — sim senza GWB (verde) vs con GWB A=-14.0, γ=13/3 (viola chiaro), stesso seed', fontsize=13, y=1.00)
fig.savefig('corner_sim_confronto_gwb_A140_g433_no_gwb.png', dpi=150, bbox_inches='tight')

In [ ]:
import corner

truths_dict = {
    'J1640+2224_dm_gp_gamma': 6.471070651479037,
    'J1640+2224_dm_gp_log10_A': -15.7227863246785,
    'J1640+2224_red_noise_gamma': 0.13235259553929019,
    'J1640+2224_red_noise_log10_A': -13.212902548624186,
    'J1640+2224_sv_gp_gamma': 1.3259815720503758,
    'J1640+2224_sv_gp_log10_A': -15.888300265911965,
}
truth_values = [truths_dict[p] for p in logL.params]
clean_labels = [name.replace('J1640+2224_', '') for name in logL.params]

# dati reali DR2low
fig = corner.corner(
    chain,
    labels=clean_labels,
    truths=truth_values,
    truth_color='#222222',
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 11},
    hist_kwargs={'density': True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    bins=40,
    color='#2E86AB',
)

# GWB A=-14.0, gamma=13/3
corner.corner(
    chain_sim_gwb5,
    labels=clean_labels,
    truths=truth_values,
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 11},
    hist_kwargs={'density': True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    bins=40,
    color='#A06CD5',
    fig=fig
)

fig.set_size_inches(16, 16)
fig.suptitle('J1640+2224 — DR2low reali (blu) vs sim con GWB A=-14.0, γ=13/3 (viola chiaro)', fontsize=13, y=1.00)
fig.savefig('corner_reali_vs_simGWB_A140_g433.png', dpi=150, bbox_inches='tight')

## Riepilogo

_Da compilare dopo il run._

- Assorbimento del GWB da parte del red noise: …
- Effetto della pendenza (γ = 2.7 vs 13/3) a parità di ampiezza: …
- Effetto dell'ampiezza (−14.5 vs −14.0) a parità di pendenza: …
- Presenza della degenerazione a 'L' tra i parametri di DM e scattering: …